# Phase 6: Statistical Analysis

Hypothesis tests comparing post-day vs non-post-day behavior in oil markets. Includes Welch's t-test for volatility, binomial tests for directional accuracy, Granger causality, Mann-Whitney U for volume anomalies, and correlation analysis.

## Load Data and Define Groups

Split the master timeline into Iran post days and non-post days.

In [10]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed'

In [11]:
master = pd.read_csv(os.path.join(PROCESSED, 'master.csv'))
master['date'] = pd.to_datetime(master['date'])
master = master.dropna(subset=['daily_return'])

post_days = master[master['is_iran_post_day'] == 1]
nonpost_days = master[master['is_iran_post_day'] == 0]

print(f"Total trading days: {len(master)}")
print(f"Iran post days: {len(post_days)}")
print(f"Non-post days: {len(nonpost_days)}")

Total trading days: 350
Iran post days: 291
Non-post days: 59


## Test 1: Post-day vs Non-post-day Volatility (Welch's t-test)

Compare absolute daily returns between the two groups to test whether Iran post days exhibit significantly higher volatility.

In [12]:
post_vol = post_days['abs_return'].dropna()
nonpost_vol = nonpost_days['abs_return'].dropna()

t_stat, p_value = stats.ttest_ind(post_vol, nonpost_vol, equal_var=False)

print(f"Post-day mean abs return:     {post_vol.mean():.4f}%")
print(f"Non-post-day mean abs return: {nonpost_vol.mean():.4f}%")
print(f"Ratio: {post_vol.mean() / nonpost_vol.mean():.2f}x")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at p<0.05: {'YES' if p_value < 0.05 else 'NO'}")
print(f"Significant at p<0.01: {'YES' if p_value < 0.01 else 'NO'}")

Post-day mean abs return:     1.7390%
Non-post-day mean abs return: 1.2327%
Ratio: 1.41x
t-statistic: 2.6640
p-value: 0.008821
Significant at p<0.05: YES
Significant at p<0.01: YES


## Test 2: Directional Accuracy (Binomial test)

Test whether escalation posts predict price increases and de-escalation posts predict price decreases at rates significantly above chance (50%).

In [5]:
esc_days = master[(master['post_direction'] == 'escalation') & master['daily_return'].notna()]
deesc_days = master[(master['post_direction'] == 'de-escalation') & master['daily_return'].notna()]

# Escalation -> price increase
esc_up = (esc_days['daily_return'] > 0).sum()
esc_total = len(esc_days)
if esc_total > 0:
    esc_pct = esc_up / esc_total
    esc_pval = stats.binomtest(esc_up, esc_total, 0.5, alternative='greater').pvalue
    print(f"Escalation posts -> price UP:  {esc_up}/{esc_total} ({esc_pct:.1%})")
    print(f"  Binomial p-value: {esc_pval:.6f}")
    print(f"  Significant at p<0.05: {'YES' if esc_pval < 0.05 else 'NO'}")

# De-escalation -> price decrease
deesc_down = (deesc_days['daily_return'] < 0).sum()
deesc_total = len(deesc_days)
if deesc_total > 0:
    deesc_pct = deesc_down / deesc_total
    deesc_pval = stats.binomtest(deesc_down, deesc_total, 0.5, alternative='greater').pvalue
    print(f"\nDe-escalation posts -> price DOWN: {deesc_down}/{deesc_total} ({deesc_pct:.1%})")
    print(f"  Binomial p-value: {deesc_pval:.6f}")
    print(f"  Significant at p<0.05: {'YES' if deesc_pval < 0.05 else 'NO'}")

Escalation posts -> price UP:  122/213 (57.3%)
  Binomial p-value: 0.019787
  Significant at p<0.05: YES

De-escalation posts -> price DOWN: 12/21 (57.1%)
  Binomial p-value: 0.331812
  Significant at p<0.05: NO


## Test 3: Granger Causality (Posts -> Price Changes)

Test whether Iran post counts Granger-cause daily oil returns at lags of 1-3 days.

In [13]:
granger_df = master[['daily_return', 'iran_posts']].dropna().reset_index(drop=True)

if len(granger_df) > 20:
    try:
        result = grangercausalitytests(granger_df[['daily_return', 'iran_posts']], maxlag=3, verbose=False)
        for lag in [1, 2, 3]:
            f_stat = result[lag][0]['ssr_ftest'][0]
            p_val = result[lag][0]['ssr_ftest'][1]
            print(f"Lag {lag}: F={f_stat:.4f}, p={p_val:.6f} {'*' if p_val < 0.05 else ''}")
    except Exception as e:
        print(f"Granger test failed: {e}")
else:
    print("Not enough observations for Granger test")

Lag 1: F=1.8708, p=0.172276 
Lag 2: F=1.8017, p=0.166574 
Lag 3: F=1.3414, p=0.260765 


## Test 4: Volume Anomaly (Mann-Whitney U)

Test whether volatility z-scores are significantly higher on Iran post days using a non-parametric rank test.

In [14]:
post_z = post_days['volume_anomaly_z'].dropna()
nonpost_z = nonpost_days['volume_anomaly_z'].dropna()

if len(post_z) > 0 and len(nonpost_z) > 0:
    u_stat, u_pval = stats.mannwhitneyu(post_z, nonpost_z, alternative='greater')
    print(f"Post-day mean z-score:     {post_z.mean():.4f}")
    print(f"Non-post-day mean z-score: {nonpost_z.mean():.4f}")
    print(f"U-statistic: {u_stat:.4f}")
    print(f"p-value: {u_pval:.6f}")
    print(f"Significant at p<0.05: {'YES' if u_pval < 0.05 else 'NO'}")

Post-day mean z-score:     0.0504
Non-post-day mean z-score: -0.1619
U-statistic: 9936.0000
p-value: 0.028297
Significant at p<0.05: YES


## Test 5: Feature Correlations

Compute and review the correlation matrix across all key features, focusing on relationships with composite_score and abs_return.

In [15]:
corr_cols = ['daily_return', 'abs_return', 'iran_posts', 'escalation_count',
             'deescalation_count', 'volume_anomaly_z', 'oscillation_score',
             'fabrication_score', 'causality_score', 'composite_score']

available_cols = [c for c in corr_cols if c in master.columns]
corr_matrix = master[available_cols].corr()

# Key correlations
print("Key correlations with composite_score:")
if 'composite_score' in corr_matrix.columns:
    for col in available_cols:
        if col != 'composite_score':
            r = corr_matrix.loc[col, 'composite_score']
            print(f"  {col:<25} r = {r:>7.4f}")

print("\nKey correlations with abs_return:")
if 'abs_return' in corr_matrix.columns:
    for col in ['iran_posts', 'escalation_count', 'composite_score']:
        if col in corr_matrix.index:
            r = corr_matrix.loc[col, 'abs_return']
            print(f"  {col:<25} r = {r:>7.4f}")

# Save correlation matrix
corr_matrix.to_csv(os.path.join(PROCESSED, 'correlation_matrix.csv'))

Key correlations with composite_score:
  daily_return              r =  0.1866
  abs_return                r =  0.5397
  iran_posts                r =  0.2866
  escalation_count          r =  0.2569
  deescalation_count        r =  0.2348
  volume_anomaly_z          r =  0.5150
  oscillation_score         r =  0.4657
  fabrication_score         r =  0.5765
  causality_score           r =  0.7329

Key correlations with abs_return:
  iran_posts                r =  0.1192
  escalation_count          r =  0.0975
  composite_score           r =  0.5397


## Summary of Statistical Findings

In [16]:
print("SUMMARY OF STATISTICAL FINDINGS")
print("=" * 60)
print(f"1. Post-day volatility: {post_vol.mean()/nonpost_vol.mean():.2f}x higher (p={p_value:.4f})")
if esc_total > 0:
    print(f"2. Escalation -> price up: {esc_pct:.1%} (p={esc_pval:.4f})")
if deesc_total > 0:
    print(f"3. De-escalation -> price down: {deesc_pct:.1%} (p={deesc_pval:.4f})")
print(f"4. Volume anomaly on post days: z={post_z.mean():.3f} (p={u_pval:.4f})")

SUMMARY OF STATISTICAL FINDINGS
1. Post-day volatility: 1.41x higher (p=0.0088)
2. Escalation -> price up: 57.3% (p=0.0198)
3. De-escalation -> price down: 57.1% (p=0.3318)
4. Volume anomaly on post days: z=0.050 (p=0.0283)
